# §0.1 線形代数の復習 - 情報幾何への橋渡し

## 1. 概要

- **この節で学ぶこと**: 内積・計量・正定値行列・双対空間の概念を復習し、Fisher情報行列への接続を準備する
- **前提知識**: 行列演算、固有値・固有ベクトル、ベクトル空間の基礎
- **情報幾何との関連**: Fisher情報行列はリーマン計量となり、自然勾配は双対変換で定義される

## 2. 直感的理解

### 計量とは「ものさし」

普段使う「距離」はユークリッド距離：$d = \sqrt{(x_2-x_1)^2 + (y_2-y_1)^2}$

しかし、空間によっては「ものさしの目盛りが場所によって違う」ことがある。

**例：地図上の距離**
- メルカトル図法では、赤道付近と極付近で同じ1cmが異なる実距離を表す
- 統計的多様体でも、パラメータの「場所」によって距離の測り方が変わる

### 双対空間とは「勾配の住む場所」

- ベクトル（方向）と勾配（傾き）は別物
- 勾配を「正しい方向」に変換するには計量が必要
- これが自然勾配法の核心

## 3. 数学的定義

### 3.1 内積と計量

**標準内積**
$$\langle u, v \rangle = u^\top v = \sum_{i=1}^n u_i v_i$$

**一般の内積（計量）**

正定値対称行列 $G$ を用いた内積：
$$\langle u, v \rangle_G = u^\top G v$$

### 3.2 正定値行列

**定義**: 対称行列 $G$ が**正定値**であるとは：
$$v^\top G v > 0 \quad (\forall v \neq 0)$$

**同値条件**:
- すべての固有値が正
- $G = L L^\top$ と分解可能（Cholesky分解）

### 3.3 双対空間

**定義**: ベクトル空間 $V$ の双対空間 $V^*$ は、$V$ から $\mathbb{R}$ への線形写像の集合

**計量による同一視**:
$$v \in V \mapsto \phi_v \in V^* \quad \text{where} \quad \phi_v(w) = v^\top G w$$

座標で書くと：
- ベクトル（反変）: $v^i$（上付き添字）
- 双対ベクトル（共変）: $v_i = G_{ij} v^j$（下付き添字）

In [ ]:
# 必要なライブラリのインポート
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

plt.rcParams['figure.figsize'] = (10, 6)

## 4. 可視化

### 4.1 計量による「単位円」の変形

In [ ]:
def plot_metric_unit_circles():
    """
    異なる計量での「単位円」を比較
    ||v||² = v^T G v = 1 となるベクトルの集合
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    theta = np.linspace(0, 2*np.pi, 100)
    
    metrics = [
        (np.eye(2), 'Identity (Euclidean)', 'blue'),
        (np.array([[2, 0], [0, 1]]), 'G = diag(2, 1)', 'red'),
        (np.array([[2, 0.5], [0.5, 1]]), 'G with off-diagonal', 'green'),
    ]
    
    for ax, (G, title, color) in zip(axes, metrics):
        # G^{-1/2} を計算して単位円を変換
        eigenvalues, eigenvectors = np.linalg.eigh(G)
        
        # 楕円のパラメータ
        angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
        width = 2 / np.sqrt(eigenvalues[0])
        height = 2 / np.sqrt(eigenvalues[1])
        
        ellipse = Ellipse((0, 0), width, height, angle=angle,
                          fill=False, edgecolor=color, linewidth=2)
        ax.add_patch(ellipse)
        
        ax.set_xlim(-2, 2)
        ax.set_ylim(-2, 2)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.axhline(0, color='k', linewidth=0.5)
        ax.axvline(0, color='k', linewidth=0.5)
        ax.set_title(f'{title}\n||v||_G = 1', fontsize=11)
        ax.set_xlabel('$v_1$')
        ax.set_ylabel('$v_2$')
    
    plt.tight_layout()
    plt.show()

plot_metric_unit_circles()

### 4.2 勾配と自然勾配の違い

In [ ]:
def visualize_natural_gradient():
    """
    通常の勾配と自然勾配の違いを可視化
    自然勾配 = G^{-1} @ 通常勾配
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 計量行列（Fisher情報行列に相当）
    G = np.array([[4, 0], [0, 1]])
    G_inv = np.linalg.inv(G)
    
    # 通常の勾配
    grad = np.array([1, 1])
    # 自然勾配
    natural_grad = G_inv @ grad
    
    # 左図：ベクトルの比較
    ax1 = axes[0]
    ax1.arrow(0, 0, grad[0], grad[1], head_width=0.1, head_length=0.05, 
              fc='blue', ec='blue', linewidth=2, label='Standard gradient')
    ax1.arrow(0, 0, natural_grad[0], natural_grad[1], head_width=0.1, head_length=0.05,
              fc='red', ec='red', linewidth=2, label='Natural gradient')
    
    # 計量楕円も表示
    theta = np.linspace(0, 2*np.pi, 100)
    ellipse_x = np.cos(theta) / 2  # 1/sqrt(4)
    ellipse_y = np.sin(theta)      # 1/sqrt(1)
    ax1.plot(ellipse_x * 1.5, ellipse_y * 1.5, 'g--', alpha=0.5, label='Metric ellipse')
    
    ax1.set_xlim(-0.5, 1.5)
    ax1.set_ylim(-0.5, 1.5)
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    ax1.set_xlabel('$\\theta_1$')
    ax1.set_ylabel('$\\theta_2$')
    ax1.set_title('Standard vs Natural gradient\nG = diag(4, 1)')
    
    # 右図：最適化軌跡の比較（概念図）
    ax2 = axes[1]
    
    # 等高線（楕円形の損失関数）
    x = np.linspace(-2, 2, 100)
    y = np.linspace(-2, 2, 100)
    X, Y = np.meshgrid(x, y)
    Z = 2*X**2 + 0.5*Y**2  # 異方性のある損失
    ax2.contour(X, Y, Z, levels=10, alpha=0.5)
    
    # 勾配降下の軌跡（概念的）
    start = np.array([1.5, 1.5])
    
    # 通常勾配降下
    traj_std = [start.copy()]
    pos = start.copy()
    for _ in range(10):
        grad = np.array([4*pos[0], pos[1]])
        pos = pos - 0.1 * grad
        traj_std.append(pos.copy())
    traj_std = np.array(traj_std)
    
    # 自然勾配降下（Gを考慮）
    traj_nat = [start.copy()]
    pos = start.copy()
    G_loss = np.array([[4, 0], [0, 1]])  # 損失のヘッセ（≈Fisher）
    for _ in range(10):
        grad = np.array([4*pos[0], pos[1]])
        natural_grad = np.linalg.inv(G_loss) @ grad
        pos = pos - 0.3 * natural_grad
        traj_nat.append(pos.copy())
    traj_nat = np.array(traj_nat)
    
    ax2.plot(traj_std[:, 0], traj_std[:, 1], 'b.-', markersize=8, label='Standard GD')
    ax2.plot(traj_nat[:, 0], traj_nat[:, 1], 'r.-', markersize=8, label='Natural GD')
    ax2.plot(0, 0, 'g*', markersize=15, label='Optimum')
    
    ax2.set_xlim(-0.5, 2)
    ax2.set_ylim(-0.5, 2)
    ax2.legend()
    ax2.set_xlabel('$\\theta_1$')
    ax2.set_ylabel('$\\theta_2$')
    ax2.set_title('Optimization trajectories')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_natural_gradient()

## 5. 具体例

### 例1：計量による内積の計算

In [ ]:
# 手計算の確認
u = np.array([1, 2])
v = np.array([3, 1])
G = np.array([[2, 0], [0, 1]])

# 標準内積
inner_std = u @ v
print(f"標準内積 <u, v> = {u} · {v} = {inner_std}")
print(f"  手計算: 1×3 + 2×1 = 5 ✓")

# 計量付き内積
inner_G = u @ G @ v
print(f"\n計量付き内積 <u, v>_G = uᵀGv = {inner_G}")
print(f"  手計算: [1,2] @ [[2,0],[0,1]] @ [3,1]ᵀ")
print(f"        = [2,2] @ [3,1]ᵀ = 6 + 2 = 8 ✓")

### 例2：正定値性の確認

In [ ]:
def check_positive_definite(G, name):
    eigenvalues = np.linalg.eigvalsh(G)
    is_pd = np.all(eigenvalues > 0)
    print(f"{name}:")
    print(f"  行列 = \n{G}")
    print(f"  固有値 = {eigenvalues}")
    print(f"  正定値: {is_pd}")
    return is_pd

# 正定値の例
G1 = np.array([[2, 1], [1, 2]])
check_positive_definite(G1, "G1")

print()

# 正定値でない例
G2 = np.array([[1, 2], [2, 1]])
check_positive_definite(G2, "G2")

### 例3：双対変換（自然勾配の計算）

In [ ]:
# 正規分布のFisher情報行列（σ=1の場合）
sigma = 1
I_fisher = np.array([[1/sigma**2, 0], 
                     [0, 2/sigma**2]])

# 通常の勾配（損失関数の微分）
grad = np.array([1, 1])

# 自然勾配 = I^{-1} @ grad
I_inv = np.linalg.inv(I_fisher)
natural_grad = I_inv @ grad

print("Fisher情報行列 I:")
print(I_fisher)
print(f"\n通常勾配: {grad}")
print(f"自然勾配: {natural_grad}")
print(f"\n解釈: μ方向はσ²={sigma**2}倍、σ方向はσ²/2={sigma**2/2}倍に拡大")

## 6. 他の概念との関係

### 前の節との繋がり
- (これが最初の節のため該当なし)

### 次の節への接続
- **確率・統計 (§0.2)**: 指数型分布族のパラメータ空間で計量が定義される
- **KLダイバージェンス (§0.3)**: KLの2次近似がFisher計量を導く

### 情報幾何への応用

| 線形代数の概念 | 情報幾何での対応 |
|--------------|----------------|
| 内積 $\langle u, v \rangle_G$ | リーマン計量 |
| 正定値行列 | Fisher情報行列の性質 |
| 二次形式の楕円 | 計量楕円（推定精度の可視化） |
| 双対空間 | 接空間と余接空間 |
| 計量による双対変換 | **自然勾配** |

## 7. 演習問題

### Q1. 計量と内積

計量行列 $G = \begin{pmatrix} 4 & 0 \\ 0 & 1 \end{pmatrix}$ に対して、
ベクトル $v = (1, 2)^\top$ の長さ $||v||_G$ を計算せよ。

<details>
<summary>解答を見る</summary>

$$||v||_G = \sqrt{v^\top G v} = \sqrt{(1, 2) \begin{pmatrix} 4 & 0 \\ 0 & 1 \end{pmatrix} \begin{pmatrix} 1 \\ 2 \end{pmatrix}}$$

$$= \sqrt{(4, 2) \begin{pmatrix} 1 \\ 2 \end{pmatrix}} = \sqrt{4 + 4} = \sqrt{8} = 2\sqrt{2}$$

</details>

In [ ]:
# Q1の検証
G = np.array([[4, 0], [0, 1]])
v = np.array([1, 2])
norm_G = np.sqrt(v @ G @ v)
print(f"||v||_G = {norm_G:.4f} = 2√2 = {2*np.sqrt(2):.4f} ✓")

### Q2. 自然勾配の計算

通常の勾配が $\nabla L = (2, 4)^\top$ のとき、
Fisher情報行列 $I = \begin{pmatrix} 2 & 0 \\ 0 & 8 \end{pmatrix}$ に対する
自然勾配を計算せよ。

<details>
<summary>解答を見る</summary>

$$\tilde{\nabla} L = I^{-1} \nabla L = \begin{pmatrix} 1/2 & 0 \\ 0 & 1/8 \end{pmatrix} \begin{pmatrix} 2 \\ 4 \end{pmatrix} = \begin{pmatrix} 1 \\ 0.5 \end{pmatrix}$$

</details>

In [ ]:
# Q2の検証
I = np.array([[2, 0], [0, 8]])
grad = np.array([2, 4])
natural_grad = np.linalg.inv(I) @ grad
print(f"自然勾配 = {natural_grad}")

### Q3. 正定値行列の判定

以下の行列が正定値かどうか判定せよ：
$$A = \begin{pmatrix} 3 & 1 \\ 1 & 2 \end{pmatrix}$$

<details>
<summary>解答を見る</summary>

固有値を計算：
$$\det(A - \lambda I) = (3-\lambda)(2-\lambda) - 1 = \lambda^2 - 5\lambda + 5 = 0$$
$$\lambda = \frac{5 \pm \sqrt{5}}{2} \approx 3.62, 1.38$$

両方正なので**正定値**。

</details>

In [ ]:
# Q3の検証
A = np.array([[3, 1], [1, 2]])
eigenvalues = np.linalg.eigvalsh(A)
print(f"固有値: {eigenvalues}")
print(f"正定値: {np.all(eigenvalues > 0)}")

## 8. 参考：使用したプロンプト

このノートブックを作成・理解する際に有効なプロンプト：

```
線形代数の「計量」の概念を、以下の形式で説明してください：
1. 数式を使わない直感的説明
2. 正規分布のFisher情報行列との対応
3. Pythonでの可視化コード（計量楕円の描画）
```

```
自然勾配法において、なぜFisher情報行列の逆行列をかけるのか、
双対空間の観点から説明してください。
カルマンフィルタの更新式との類似点があれば指摘してください。
```

```
正定値行列の3つの同値条件（固有値、Cholesky分解、主小行列式）を
具体的な2×2行列で確認するPythonコードを書いてください。
```

---
**次のノートブック**: `02_probability_statistics.ipynb` - 確率・統計の復習